# Annotation

In [2]:
import warnings

warnings.filterwarnings("ignore", category=DeprecationWarning)

from numba.core.errors import NumbaDeprecationWarning

warnings.simplefilter("ignore", category=NumbaDeprecationWarning)

In [3]:
import urllib.request
from pathlib import Path

import celltypist
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc

# the environment variable SCIPY_ARRAY_API needs to be set to 1 before importing either Scikit-learn or SciPy
import os
os.environ['SCIPY_ARRAY_API']='1'

import scarches as sca
import seaborn as sns
from celltypist import models
from scipy.sparse import csr_matrix

warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)

sc.set_figure_params(figsize=(5, 5))

In [4]:
import anndata
adata = anndata.read_h5ad("../clustering/clustering.h5ad.gz")
adata

AnnData object with n_obs × n_vars = 892 × 236796
    obs: 'sample', 'fastq_1', 'fastq_2', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_20_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'log1p_total_counts_hb', 'pct_counts_hb', 'outlier', 'mt_outlier', 'scDblFinder_score', 'scDblFinder_class', 'size_factors', 'leiden_res0_25', 'leiden_res0_5', 'leiden_res1', 'leiden_res1_25'
    var: 'gene_versions', 'gene_symbol', 'gene_name', 'mt', 'ribo', 'hb', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'highly_deviant', 'binomial_deviance', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'hvg', 'leiden_res0_25', 'leiden_res0_25_colors', 'leiden_res0_5', 'leiden_res0_5_colors', 'leiden_res1', 'leiden_res1_25', 'leiden_res1_25_colors', '

## CellTypist automated annotation using Adult_Human_Skin model

In [22]:
adata_celltypist = adata.copy()  # make a copy of our adata
adata_celltypist.X = adata.layers["counts"]  # set adata.X to raw counts
sc.pp.normalize_total(
    adata_celltypist, target_sum=10**4
)  # normalize to 10,000 counts per cell
sc.pp.log1p(adata_celltypist)  # log-transform
# make .X dense instead of sparse, for compatibility with celltypist:
adata_celltypist.X = adata_celltypist.X.toarray()

In [23]:
models.download_models(
    force_update=True, model=["Adult_Human_Skin.pkl"]
)

📜 Retrieving model list from server https://celltypist.cog.sanger.ac.uk/models/models.json
📚 Total models in list: 54
📂 Storing models in /home/philip/.celltypist/data/models
💾 Total models to download: 1
💾 Downloading model [1/1]: Adult_Human_Skin.pkl


In [24]:
model = models.Model.load(model="Adult_Human_Skin.pkl")

In [25]:
model.cell_types

array(['DC1', 'DC2', 'Differentiated_KC', 'F1', 'F2', 'F3', 'ILC1_3',
       'ILC1_NK', 'ILC2', 'Inf_mac', 'LC', 'LE1', 'LE2', 'Macro_1',
       'Macro_2', 'Mast_cell', 'Melanocyte', 'MigDC', 'Mono_mac', 'NK',
       'Pericyte_1', 'Pericyte_2', 'Plasma', 'Schwann_1', 'Schwann_2',
       'Tc', 'Th', 'Treg', 'Undifferentiated_KC', 'VE1', 'VE2', 'VE3',
       'migLC', 'moDC'], dtype=object)

In [11]:
adata_celltypist.var_names = adata_celltypist.var['gene_symbol']

In [28]:
na_genes = adata_celltypist.var['gene_symbol'].isna()
adata_celltypist.var = adata_celltypist.var.loc[~(na_genes), :]
adata_celltypist.X = adata_celltypist.X[:, np.logical_not(na_genes)]


ValueError: Length of passed value for var_names is 56134, but this AnnData has shape: (892, 236796)

In [12]:
predictions = celltypist.annotate(
    adata_celltypist, model=model, majority_voting=True
)

TypeError: unsupported operand type(s) for +: 'float' and 'str'